
<div style="background: linear-gradient(135deg,#0d1117,#1c2235); padding:50px 40px; border-radius:16px; text-align:center; color:white; margin-bottom:20px;">
  <div style="font-size:3rem; margin-bottom:8px;">₿ Ξ</div>
  <h1 style="font-size:2.4rem; margin:0 0 10px; letter-spacing:-1px;">Crypto Market Dashboard</h1>
  <h3 style="color:#7eb8f7; font-weight:400; margin:0 0 20px;">Informe Analítico Completo</h3>
  <p style="color:rgba(255,255,255,0.7); font-size:1rem; margin:0;">
    Análisis de mercado · Series temporales · Estacionariedad ·
    Descomposición · ARIMA · Correlaciones
  </p>
  <hr style="border-color:rgba(255,255,255,0.15); margin:24px 0;">
  <p style="color:rgba(255,255,255,0.5); font-size:0.85rem; margin:0;">
    API: CryptoCompare &nbsp;|&nbsp; Lenguaje: Python 3.11
    &nbsp;|&nbsp; Librerías: Dash · Plotly · Statsmodels · Scipy
  </p>
</div>

---

## 📋 Tabla de Contenidos

| # | Sección | Descripción |
|---|---------|-------------|
| 1 | [Introducción](#intro) | Contexto y fuente de datos |
| 2 | [Objetivos](#objetivos) | General y específicos |
| 3 | [Marco Teórico](#marco) | Conceptos clave |
| 4 | [Datos del mercado](#datos) | Exploración inicial |
| 5 | [Análisis exploratorio (EDA)](#eda) | Precios, retornos, volatilidad |
| 6 | [Estacionariedad](#estac) | ADF y boxplots por período |
| 7 | [Descomposición](#decomp) | Aditiva vs multiplicativa |
| 8 | [Correlaciones](#corr) | Pearson entre las 10 monedas |
| 9 | [Predicción ARIMA](#arima) | Modelo, diagnósticos, pronóstico |
| 10 | [Conclusiones](#conclusiones) | Hallazgos principales |

---



<a id="intro"></a>
## 1. Introducción

Este informe documenta el análisis completo implementado en el **Crypto Market Dashboard**, 
una aplicación interactiva construida con Python y Dash que consume datos en tiempo real 
de la API **CryptoCompare**.

### 1.1 ¿Qué es CryptoCompare?

CryptoCompare es una de las fuentes de datos de criptomonedas más utilizadas del mundo.  
Provee datos históricos OHLCV (Open, High, Low, Close, Volume) con granularidad diaria, 
horaria y por minuto, así como snapshots en tiempo real de market cap y volumen.

**Endpoints utilizados en este proyecto:**

| Endpoint | Descripción | Datos obtenidos |
|----------|-------------|-----------------|
| `/data/top/mktcapfull` | Top N por market cap | Precio, MCap, Volumen, Cambio 24h |
| `/data/v2/histoday` | Histórico diario | OHLCV hasta 2 000 días |

### 1.2 Monedas analizadas

Las **10 principales criptomonedas** por capitalización de mercado:

| Símbolo | Nombre | Categoría |
|---------|--------|-----------|
| BTC | Bitcoin | Large Cap |
| ETH | Ethereum | Large Cap |
| BNB | BNB | Large Cap |
| SOL | Solana | Mid Cap |
| XRP | XRP | Mid Cap |
| ADA | Cardano | Mid Cap |
| AVAX | Avalanche | Mid Cap |
| DOGE | Dogecoin | Mid Cap |
| LTC | Litecoin | Small Cap |
| LINK | Chainlink | Small Cap |

### 1.3 Sistema de caché

Para no exceder los límites de la API gratuita (~50 req/hora sin API key), 
se implementó un sistema de caché en memoria con TTL de **5 minutos** para datos de 
snapshot y **1 hora** para datos históricos. Si la API falla, el sistema activa 
un **fallback sintético** que genera datos realistas con semilla única por moneda.


In [1]:

# ── Instalación de dependencias (ejecutar una sola vez) ──────────────────────
# !pip install requests pandas numpy plotly statsmodels scipy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats as sp_stats
import requests, hashlib, time
from datetime import datetime, timedelta

print("✅ Librerías cargadas correctamente")
print(f"   NumPy     {np.__version__}")
print(f"   Pandas    {pd.__version__}")
import plotly.offline as pyo
pyo.init_notebook_mode(connected=True)

✅ Librerías cargadas correctamente
   NumPy     1.26.4
   Pandas    2.2.2



<a id="objetivos"></a>
## 2. Objetivos del Proyecto

### 2.1 Objetivo General

Desarrollar un dashboard analítico interactivo en tiempo real que integre datos de 
CryptoCompare para explorar el mercado cripto mediante análisis de series temporales, 
descomposición aditiva y multiplicativa, pruebas de estacionariedad y predicción ARIMA.

### 2.2 Objetivos Específicos

1. **Consumir datos en tiempo real** — Integrar la API CryptoCompare con caché y fallback.
2. **Analizar series de tiempo OHLCV** — Velas japonesas, SMA, retornos y volatilidad.
3. **Evaluar estacionariedad** — Prueba ADF y visualización con boxplots por trimestre.
4. **Descomposición aditiva y multiplicativa** — Separar Tendencia, Estacionalidad y Residuo.
5. **Calcular correlaciones** — Matriz de Pearson entre las 10 monedas.
6. **Predecir con ARIMA** — Pronóstico con intervalos de confianza y diagnósticos.

### 2.3 Justificación

| Dimensión | Argumento |
|-----------|-----------|
| **Económica** | El mercado cripto supera los $2 billones USD. La volatilidad hace necesarias herramientas analíticas para decisiones informadas. |
| **Técnica** | Integra APIs REST, estadística, pruebas de hipótesis, descomposición de series y modelos ARIMA en una sola aplicación open-source. |
| **Académica** | Aplica conceptos de series temporales financieras adaptables a stocks, forex o commodities. |



<a id="marco"></a>
## 3. Marco Teórico

### 3.1 Conceptos del Mercado Cripto

**Blockchain:** Red descentralizada de bloques encadenados. Cada bloque contiene el hash 
del anterior, garantizando integridad sin intermediarios.

**Market Cap:** `Market Cap = Precio × Unidades en circulación`. Principal métrica de tamaño.

**OHLCV:** Open, High, Low, Close, Volume — los cinco datos que describen el comportamiento 
de un activo en un período. Base del análisis técnico.

**Volatilidad:**

$$\sigma_{anual} = \sigma_{diaria} \times \sqrt{365}$$

Bitcoin presenta volatilidad histórica del 60–80% anual frente al 15–20% de acciones tradicionales.

---

### 3.2 Series Temporales y Estacionariedad

Una serie temporal es una secuencia de observaciones registradas en intervalos regulares.

**Retornos diarios:**
$$r_t = \left(\frac{P_t}{P_{t-1}} - 1\right) \times 100$$

**Estacionariedad:** Una serie es estacionaria si su media, varianza y autocovarianza 
no cambian con el tiempo. Es requisito fundamental para ARIMA.

| Serie | ¿Estacionaria? | Por qué |
|-------|---------------|---------|
| Precios |  NO | Tienen tendencia creciente de largo plazo |
| Retornos |  SÍ | Fluctúan alrededor de ≈0% sin tendencia |

---

### 3.3 Descomposición: Aditiva vs Multiplicativa

Toda serie puede separarse en: **Tendencia (T)**, **Estacionalidad (S)** y **Residuo (R)**.

#### Modelo ADITIVO
$$Y_t = T_t + S_t + R_t$$

- Usar cuando la **amplitud de la estacionalidad es CONSTANTE** con el tiempo.
- Cada componente se suma directamente en las mismas unidades (USD).
- **Ejemplo:** temperatura mensual, datos sin crecimiento exponencial.

#### Modelo MULTIPLICATIVO
$$Y_t = T_t \times S_t \times R_t$$

- Usar cuando la **amplitud crece proporcionalmente** al nivel de la serie.
- Los componentes son factores (S=1 = sin efecto estacional, S>1 = efecto positivo).
- **Aplicación en cripto:** Los precios crecen exponencialmente, por lo que una oscilación 
  del 5% representa $50 cuando BTC vale $1 000, pero $2 500 cuando vale $50 000.

**¿Por qué multiplicativo en cripto?**

Si los cambios son **proporcionales** (expresados en %) → multiplicativo.  
Si los cambios son **absolutos** (expresados en USD fijos) → aditivo.  

Los retornos logarítmicos convierten el modelo multiplicativo en aditivo:
$$\log(Y_t) = \log(T_t) + \log(S_t) + \log(R_t)$$

---

### 3.4 Modelo ARIMA(p, d, q)

**AutoRegressive Integrated Moving Average:**

- **p** = orden autoregresivo (cuántos valores pasados se usan como predictores)
- **d** = número de diferenciaciones para lograr estacionariedad (en precios cripto: d=1)
- **q** = orden de media móvil (cuántos errores pasados se incluyen)

**Expresión matemática:**
$$\Delta^d Y_t = c + \sum_{i=1}^{p} \phi_i \Delta^d Y_{t-i} + \varepsilon_t + \sum_{j=1}^{q} \theta_j \varepsilon_{t-j}$$

**Criterios de selección del orden:**

| Criterio | Fórmula | Interpretación |
|----------|---------|---------------|
| AIC | $-2\log L + 2k$ | Menor = mejor balance ajuste/complejidad |
| BIC | $-2\log L + k\log n$ | Penaliza más la complejidad que AIC |

---

### 3.5 Pruebas Estadísticas

**ADF (Augmented Dickey-Fuller):**
- $H_0$: La serie tiene raíz unitaria (NO estacionaria)
- $p > 0.05$ → No rechazamos $H_0$ → NO estacionaria
- $p \leq 0.05$ → Rechazamos $H_0$ → SÍ estacionaria

**Jarque-Bera:**
- $H_0$: Los retornos siguen distribución normal
- En cripto: casi siempre $p < 0.05$ → NO son normales (colas gruesas)

**Ljung-Box (diagnóstico ARIMA):**
- $H_0$: Los residuos son ruido blanco (independientes)
- $p > 0.05$ → Modelo adecuado
- $p \leq 0.05$ → Hay estructura residual sin capturar

---

### 3.6 Tabla de Operacionalización de Variables


In [2]:

# Tabla de operacionalización de variables del proyecto
variables = {
    "Variable": ["price","market_cap","volume_24h","change_pct_24h",
                 "high_24h","low_24h","close","retorno_diario",
                 "sma_7","sma_30","vol_anual_30d",
                 "tendencia","estacionalidad","residuo","fc_arima"],
    "Tipo": ["Cuantitativa continua"]*15,
    "Definición operacional": [
        "Precio actual en USD",
        "Precio × unidades en circulación",
        "Valor total transaccionado en 24 h",
        "Variación % del precio en 24 h",
        "Precio máximo en 24 h",
        "Precio mínimo en 24 h",
        "Precio de cierre diario",
        "(P_t / P_{t-1} − 1) × 100",
        "Media simple de los últimos 7 cierres",
        "Media simple de los últimos 30 cierres",
        "σ_diaria × √365 con ventana 30 días",
        "Componente de largo plazo (STL)",
        "Componente periódico (STL)",
        "Componente irregular (STL)",
        "Precio pronosticado por ARIMA(p,d,q)",
    ],
    "Unidad": ["USD","USD","USD","%","USD","USD","USD",
               "%","USD","USD","% anual","USD","USD/%","USD/%","USD"],
    "Rol": ["Descriptiva","Descriptiva","Predictora","Predictora",
            "Indicador técnico","Indicador técnico","Target ARIMA",
            "Feature derivada","Indicador técnico","Indicador técnico",
            "Feature derivada","Descomposición","Descomposición",
            "Descomposición","Variable objetivo"],
}

df_vars = pd.DataFrame(variables)
print("Tabla de operacionalización de variables:")
print("=" * 80)
print(df_vars.to_string(index=False))


Tabla de operacionalización de variables:
      Variable                  Tipo                 Definición operacional  Unidad               Rol
         price Cuantitativa continua                   Precio actual en USD     USD       Descriptiva
    market_cap Cuantitativa continua       Precio × unidades en circulación     USD       Descriptiva
    volume_24h Cuantitativa continua     Valor total transaccionado en 24 h     USD        Predictora
change_pct_24h Cuantitativa continua         Variación % del precio en 24 h       %        Predictora
      high_24h Cuantitativa continua                  Precio máximo en 24 h     USD Indicador técnico
       low_24h Cuantitativa continua                  Precio mínimo en 24 h     USD Indicador técnico
         close Cuantitativa continua                Precio de cierre diario     USD      Target ARIMA
retorno_diario Cuantitativa continua              (P_t / P_{t-1} − 1) × 100       %  Feature derivada
         sma_7 Cuantitativa continua  Me


<a id="datos"></a>
## 4. Datos del Mercado — Exploración Inicial

### 4.1 Obtención de datos

Se conecta al endpoint de CryptoCompare. Si no hay conexión, se usa el fallback sintético.


In [3]:

# ── Función de fallback sintético con semilla única por moneda ───────────────

def sym_seed(symbol: str) -> int:
    return int(hashlib.md5(symbol.encode()).hexdigest(), 16) % (2**31)

COINS_META = [
    ("BTC","Bitcoin",95000,1850e9),("ETH","Ethereum",3200,385e9),
    ("BNB","BNB",650,95e9),("SOL","Solana",185,87e9),
    ("XRP","XRP",2.50,143e9),("ADA","Cardano",1.10,39e9),
    ("AVAX","Avalanche",40,17e9),("DOGE","Dogecoin",0.38,56e9),
    ("LTC","Litecoin",95,7e9),("LINK","Chainlink",20,12e9),
]

BASE_PRICES = {s:p for s,_,p,_ in COINS_META}

def get_snapshot() -> pd.DataFrame:
    """Intenta API real; fallback a sintético."""
    try:
        r = requests.get(
            "https://min-api.cryptocompare.com/data/top/mktcapfull",
            params={"limit":10,"tsym":"USD","page":0},
            timeout=8
        )
        data = r.json()
        if data.get("Response") == "Success":
            rows = []
            for item in data.get("Data",[]):
                coin = item.get("CoinInfo",{})
                raw  = item.get("RAW",{}).get("USD",{})
                if raw:
                    rows.append({
                        "symbol":coin.get("Name",""),
                        "name":coin.get("FullName",""),
                        "price":float(raw.get("PRICE",0)),
                        "market_cap":float(raw.get("MKTCAP",0)),
                        "volume_24h":float(raw.get("VOLUME24HOURTO",0)),
                        "change_pct_24h":float(raw.get("CHANGEPCT24HOUR",0)),
                        "high_24h":float(raw.get("HIGH24HOUR",0)),
                        "low_24h":float(raw.get("LOW24HOUR",0)),
                    })
            if rows:
                print(" Datos obtenidos desde la API CryptoCompare")
                return pd.DataFrame(rows)
    except Exception as e:
        print(f"⚠️  API no disponible ({e}). Usando fallback sintético.")

    # ── Fallback sintético ────────────────────────────────────────────────
    rows = []
    for sym, name, base_price, base_mcap in COINS_META:
        rng   = np.random.default_rng(sym_seed(sym) + int(time.time()//300))
        noise = rng.uniform(0.93,1.07)
        price = base_price * noise
        chg   = rng.uniform(-10,10)
        rows.append({
            "symbol":sym,"name":name,"price":price,
            "market_cap":base_mcap*noise,
            "volume_24h":base_mcap*rng.uniform(0.02,0.18),
            "change_pct_24h":chg,
            "high_24h":price*rng.uniform(1.01,1.06),
            "low_24h":price*rng.uniform(0.94,0.99),
        })
    print(" Datos sintéticos generados correctamente")
    return pd.DataFrame(rows)

df_snap = get_snapshot()
print(f"\n Dataset: {df_snap.shape[0]} monedas × {df_snap.shape[1]} variables")


 Datos sintéticos generados correctamente

 Dataset: 10 monedas × 8 variables


In [4]:

# ── Estadísticas descriptivas del snapshot ───────────────────────────────────

def fmt_usd(v):
    if v>=1e12: return f"${v/1e12:.2f}T"
    if v>=1e9:  return f"${v/1e9:.2f}B"
    if v>=1e6:  return f"${v/1e6:.2f}M"
    return f"${v:,.2f}"

print("=" * 65)
print(f"{'Símbolo':<8} {'Precio':>12} {'Market Cap':>12} {'Vol 24h':>10} {'Cambio 24h':>11}")
print("-" * 65)
for _, r in df_snap.iterrows():
    p   = r["price"]
    fp  = f"${p:,.2f}" if p>=1 else f"${p:.4f}"
    sgn = "▲" if r["change_pct_24h"]>=0 else "▼"
    print(f"{r['symbol']:<8} {fp:>12} {fmt_usd(r['market_cap']):>12} "
          f"{fmt_usd(r['volume_24h']):>10} {sgn} {abs(r['change_pct_24h']):.2f}%")
print("=" * 65)
print(f"\n Capitalización Total:   {fmt_usd(df_snap['market_cap'].sum())}")
print(f" Volumen 24h Total:       {fmt_usd(df_snap['volume_24h'].sum())}")
btc_dom = df_snap.loc[df_snap['symbol']=='BTC','market_cap'].sum() / df_snap['market_cap'].sum() * 100
print(f" Dominancia BTC:          {btc_dom:.1f}%")
print(f" Cambio promedio 24h:     {df_snap['change_pct_24h'].mean():+.2f}%")


Símbolo        Precio   Market Cap    Vol 24h  Cambio 24h
-----------------------------------------------------------------
BTC        $99,919.04       $1.95T   $157.39B ▲ 9.51%
ETH         $3,294.37     $396.35B    $10.17B ▲ 6.99%
BNB           $677.47      $99.02B    $14.36B ▲ 6.49%
SOL           $186.08      $87.51B     $3.05B ▼ 6.37%
XRP             $2.41     $137.75B     $5.36B ▲ 5.54%
ADA             $1.08      $38.45B     $3.58B ▲ 7.52%
AVAX           $40.24      $17.10B     $2.09B ▲ 6.08%
DOGE          $0.3758      $55.38B     $7.67B ▲ 9.49%
LTC            $94.30       $6.95B   $477.67M ▼ 5.33%
LINK           $18.65      $11.19B     $1.50B ▼ 6.87%

 Capitalización Total:   $2.80T
 Volumen 24h Total:       $205.65B
 Dominancia BTC:          69.6%
 Cambio promedio 24h:     +3.31%


In [5]:
# ── Visualización: Market Cap y distribución ─────────────────────────────────

from plotly.subplots import make_subplots

# Gráfica de barras — Market Cap
df_sorted = df_snap.sort_values("market_cap")
colors_bar = ["#88e0b4" if c>=0 else "#f7958a"
              for c in df_sorted["change_pct_24h"]]

COLORS = ["#f7c87e","#7eb8f7","#c5b8f0","#7ed4c5",
          "#f7a8c4","#88e0b4","#f7958a","#b8d8f0","#e0c57e","#c8f0b8"]

# ── Figura 1: Barras Market Cap ──────────────────────────────────────────────
fig1 = go.Figure(go.Bar(
    x=df_sorted["market_cap"], y=df_sorted["symbol"],
    orientation="h", marker_color=colors_bar,
    text=df_sorted["market_cap"].apply(fmt_usd),
    textposition="outside", textfont=dict(size=9),
    name="Market Cap"))

fig1.update_layout(
    height=380, paper_bgcolor="#161b22", plot_bgcolor="#161b22",
    font_color="#8b949e", title_text="Market Cap — Top 10 Criptomonedas",
    title_font_color="#e6edf3", showlegend=False,
    margin=dict(t=55,b=20,l=20,r=80))
fig1.update_xaxes(showticklabels=False, gridcolor="#1c2230")
fig1.update_yaxes(tickfont=dict(color="#e6edf3"))
fig1.show()

# ── Figura 2: Dona de dominancia ─────────────────────────────────────────────
fig2 = go.Figure(go.Pie(
    labels=df_snap["symbol"], values=df_snap["market_cap"],
    hole=0.5, marker=dict(colors=COLORS),
    name="Dominancia"))

fig2.update_layout(
    height=380, paper_bgcolor="#161b22",
    font_color="#8b949e", title_text="Dominancia de mercado",
    title_font_color="#e6edf3",
    legend=dict(font=dict(color="#8b949e")),
    margin=dict(t=55,b=10,l=10,r=10))
fig2.show()

print("\nVerde = subió en 24h | Rojo = bajó en 24h")
print("La dona muestra qué porcentaje del mercado total controla cada moneda.")



Verde = subió en 24h | Rojo = bajó en 24h
La dona muestra qué porcentaje del mercado total controla cada moneda.



<a id="eda"></a>
## 5. Análisis Exploratorio de Datos (EDA)

### 5.1 Serie de tiempo histórica

Para el EDA usamos datos históricos diarios de **Bitcoin (BTC)** como referencia principal.


In [6]:

# ── Obtener histórico (API o sintético) ──────────────────────────────────────

def get_histoday(symbol="BTC", days=365) -> pd.DataFrame:
    try:
        r = requests.get("https://min-api.cryptocompare.com/data/v2/histoday",
            params={"fsym":symbol,"tsym":"USD","limit":days},timeout=12)
        data = r.json()
        if data.get("Response")=="Success":
            entries = data.get("Data",{}).get("Data",[])
            if entries:
                df = pd.DataFrame(entries)
                df["date"] = pd.to_datetime(df["time"],unit="s")
                df = df.rename(columns={"volumeto":"volume"})
                df = df[["date","open","high","low","close","volume"]]
                df = df[df["close"]>0].sort_values("date").reset_index(drop=True)
                print(f" Histórico {symbol} desde API: {len(df)} días")
                return df
    except Exception:
        pass

    # fallback sintético
    seed  = sym_seed(symbol)
    rng   = np.random.default_rng(seed)
    base  = BASE_PRICES.get(symbol, 100.0)
    drift = rng.uniform(-0.001, 0.0025)
    vol   = rng.uniform(0.022, 0.065)
    n     = days + 1
    regime = np.sin(np.linspace(0, 4*np.pi, n) + rng.uniform(0,2*np.pi)) * 0.001
    returns = rng.normal(drift, vol, n) + regime
    prices = [base]
    for rv in reversed(returns[1:]):
        prices.insert(0, max(prices[0]/(1+rv), base*0.005))
    dates  = [datetime.utcnow()-timedelta(days=days-i) for i in range(n)]
    opens  = [max(p*rng.uniform(0.985,1.005),1e-9) for p in prices]
    highs  = [max(p*rng.uniform(1.002,1.055),p)    for p in prices]
    lows   = [max(p*rng.uniform(0.945,0.998),1e-9) for p in prices]
    vols   = [abs(rng.normal(base*8000,base*2500))  for _  in prices]
    print(f" Histórico {symbol} sintético generado: {n} días")
    return pd.DataFrame({"date":dates,"open":opens,"high":highs,
                          "low":lows,"close":prices,"volume":vols})

df_btc = get_histoday("BTC", days=365)
print(f"\nPrimera fecha: {df_btc['date'].iloc[0].date()}")
print(f"Última fecha:  {df_btc['date'].iloc[-1].date()}")
print(f"Precio inicial: ${df_btc['close'].iloc[0]:,.2f}")
print(f"Precio final:   ${df_btc['close'].iloc[-1]:,.2f}")
ret_total = (df_btc['close'].iloc[-1]/df_btc['close'].iloc[0]-1)*100
print(f"Rendimiento:    {ret_total:+.1f}%")


 Histórico BTC desde API: 366 días

Primera fecha: 2025-05-12
Última fecha:  2026-05-12
Precio inicial: $102,802.44
Precio final:   $80,742.40
Rendimiento:    -21.5%


In [7]:

# ── Velas japonesas OHLC + SMA + Volumen ─────────────────────────────────────

df_btc["sma30"] = df_btc["close"].rolling(30).mean()
df_btc["sma7"]  = df_btc["close"].rolling(7).mean()

fig = make_subplots(rows=2,cols=1,shared_xaxes=True,
    row_heights=[0.75,0.25],vertical_spacing=0.02)

fig.add_trace(go.Candlestick(
    x=df_btc["date"],open=df_btc["open"],high=df_btc["high"],
    low=df_btc["low"],close=df_btc["close"],
    increasing_line_color="#88e0b4",decreasing_line_color="#f7958a",
    name="BTC",line=dict(width=1)), row=1,col=1)

fig.add_trace(go.Scatter(x=df_btc["date"],y=df_btc["sma7"],
    line=dict(color="#8b949e",width=1,dash="dot"),name="SMA 7"), row=1,col=1)

fig.add_trace(go.Scatter(x=df_btc["date"],y=df_btc["sma30"],
    line=dict(color="#7eb8f7",width=1.5),name="SMA 30"), row=1,col=1)

bc = ["#88e0b4" if c>=o else "#f7958a"
      for c,o in zip(df_btc["close"],df_btc["open"])]
fig.add_trace(go.Bar(x=df_btc["date"],y=df_btc["volume"],
    marker_color=bc,opacity=0.55,name="Volumen"), row=2,col=1)

fig.update_layout(height=500,paper_bgcolor="#161b22",plot_bgcolor="#161b22",
    font_color="#8b949e",title_text="BTC/USD — Velas OHLC + SMA 7/30 + Volumen",
    title_font_color="#e6edf3",xaxis_rangeslider_visible=False,
    margin=dict(t=55,b=20,l=60,r=20),hovermode="x unified",
    legend=dict(bgcolor="rgba(0,0,0,0)",font=dict(color="#8b949e")))
for r in [1,2]:
    fig.update_xaxes(row=r,gridcolor="#1c2230",zeroline=False,tickfont=dict(color="#8b949e"))
    fig.update_yaxes(row=r,gridcolor="#1c2230",zeroline=False,tickfont=dict(color="#8b949e"))
fig.update_yaxes(row=1,tickprefix="$",tickformat=",")
fig.show()
print("\n Verde = día positivo (cierre > apertura)")
print(" Rojo  = día negativo (cierre < apertura)")
print(" SMA 7: tendencia corto plazo | SMA 30: tendencia medio plazo")



 Verde = día positivo (cierre > apertura)
 Rojo  = día negativo (cierre < apertura)
 SMA 7: tendencia corto plazo | SMA 30: tendencia medio plazo


In [8]:

# ── Distribución de retornos diarios ─────────────────────────────────────────

rets = df_btc["close"].pct_change().dropna() * 100
mu, sig = float(rets.mean()), float(rets.std())
kurt    = float(rets.kurtosis())
skew    = float(rets.skew())

print(" Estadísticas descriptivas de retornos diarios:")
print(f"   Media (μ):           {mu:+.4f} %")
print(f"   Desv. estándar (σ):  {sig:.4f} %")
print(f"   Mínimo:              {rets.min():.2f} %")
print(f"   Máximo:              {rets.max():.2f} %")
print(f"   Curtosis:            {kurt:.4f}  {'← leptocúrtica (colas gruesas)' if kurt>3 else ''}")
print(f"   Asimetría:           {skew:.4f}")
print(f"   Volatilidad anual:   {sig * np.sqrt(365):.1f} %")

xn = np.linspace(rets.min(), rets.max(), 300)
yn = sp_stats.norm.pdf(xn, mu, sig)

fig = go.Figure()
fig.add_trace(go.Histogram(x=rets,nbinsx=55,marker_color="#7eb8f7",
    opacity=0.72,histnorm="probability density",name="Retornos BTC"))
fig.add_trace(go.Scatter(x=xn,y=yn,mode="lines",
    line=dict(color="#f7958a",width=2,dash="dot"),name="Normal referencia"))
fig.add_vline(x=0,  line_dash="dot",line_color="#484f58",line_width=1)
fig.add_vline(x=mu, line_dash="dash",line_color="#7eb8f7",line_width=1,
    annotation_text=f"μ={mu:.2f}%",annotation_font_size=9,annotation_font_color="#7eb8f7")
fig.update_layout(height=380,paper_bgcolor="#161b22",plot_bgcolor="#161b22",
    font_color="#8b949e",title_text="Distribución de retornos diarios BTC",
    title_font_color="#e6edf3",xaxis_title="Retorno diario (%)",
    yaxis_title="Densidad de probabilidad",
    legend=dict(bgcolor="rgba(0,0,0,0)",font=dict(color="#8b949e")),
    margin=dict(t=55,b=40,l=60,r=20))
fig.update_xaxes(gridcolor="#1c2230",ticksuffix="%",tickfont=dict(color="#8b949e"))
fig.update_yaxes(gridcolor="#1c2230",tickfont=dict(color="#8b949e"))
fig.show()
print("\n💡 La distribución real tiene más área en las colas que la normal (leptocurtosis)")
print("   → Más días con cambios extremos de lo que predice la teoría clásica")


 Estadísticas descriptivas de retornos diarios:
   Media (μ):           -0.0417 %
   Desv. estándar (σ):  2.2074 %
   Mínimo:              -13.98 %
   Máximo:              12.30 %
   Curtosis:            6.8068  ← leptocúrtica (colas gruesas)
   Asimetría:           -0.2374
   Volatilidad anual:   42.2 %



💡 La distribución real tiene más área en las colas que la normal (leptocurtosis)
   → Más días con cambios extremos de lo que predice la teoría clásica



<a id="estac"></a>
## 6. Análisis de Estacionariedad

### 6.1 ¿Por qué importa la estacionariedad?

Los modelos ARIMA requieren series estacionarias. Si una serie no lo es, hay que 
transformarla (diferenciarla) hasta que lo sea.

La forma visual más clara de detectar no-estacionariedad es mediante **boxplots por período**:
- Si las cajas están a **distintas alturas** → la media cambia → **NO estacionaria**
- Si las cajas están al **mismo nivel** → media constante → **SÍ estacionaria**


In [9]:

# ── Prueba ADF — Augmented Dickey-Fuller ─────────────────────────────────────

from statsmodels.tsa.stattools import adfuller

print("=" * 60)
print("PRUEBA AUGMENTED DICKEY-FULLER (ADF)")
print("H₀: La serie tiene raíz unitaria (NO estacionaria)")
print("=" * 60)

# ADF sobre PRECIOS
adf_p = adfuller(df_btc["close"].dropna(), autolag="AIC")
print(f"\n PRECIOS BTC:")
print(f"   Estadístico ADF:  {adf_p[0]:.4f}")
print(f"   p-valor:          {adf_p[1]:.6f}")
print(f"   Resultado:        {' NO rechazamos H₀ → NO estacionaria (esperado)' if adf_p[1]>0.05 else '✅ Rechazamos H₀ → Estacionaria'}")

# ADF sobre RETORNOS
adf_r = adfuller(rets.dropna(), autolag="AIC")
print(f"\n RETORNOS BTC:")
print(f"   Estadístico ADF:  {adf_r[0]:.4f}")
print(f"   p-valor:          {adf_r[1]:.6f}")
print(f"   Resultado:        {' Rechazamos H₀ → SÍ estacionaria (esperado)' if adf_r[1]<=0.05 else '⚠️ No rechazamos H₀ → NO estacionaria'}")

print("\n INTERPRETACIÓN:")
print("   Los PRECIOS no son estacionarios → se diferencian para ARIMA (d=1)")
print("   Los RETORNOS sí son estacionarios → aptos para modelos estadísticos")


PRUEBA AUGMENTED DICKEY-FULLER (ADF)
H₀: La serie tiene raíz unitaria (NO estacionaria)

 PRECIOS BTC:
   Estadístico ADF:  -0.9257
   p-valor:          0.779381
   Resultado:         NO rechazamos H₀ → NO estacionaria (esperado)

 RETORNOS BTC:
   Estadístico ADF:  -20.2348
   p-valor:          0.000000
   Resultado:         Rechazamos H₀ → SÍ estacionaria (esperado)

 INTERPRETACIÓN:
   Los PRECIOS no son estacionarios → se diferencian para ARIMA (d=1)
   Los RETORNOS sí son estacionarios → aptos para modelos estadísticos


In [10]:

# ── Boxplot por trimestre: PRECIOS vs RETORNOS ───────────────────────────────

df2 = df_btc.copy()
df2["date"]    = pd.to_datetime(df2["date"])
df2["retorno"] = df2["close"].pct_change() * 100
df2["trim"]    = df2["date"].dt.to_period("Q").astype(str)
trims          = sorted(df2["trim"].unique())[-6:]  # últimos 6 trimestres
df2            = df2[df2["trim"].isin(trims)]

fig = make_subplots(rows=1, cols=2,
    subplot_titles=["PRECIOS por trimestre (❌ NO estacionario)",
                    "RETORNOS por trimestre (✅ SÍ estacionario)"])

# Panel izquierdo: PRECIOS
for trim in trims:
    sub = df2[df2["trim"]==trim]["close"].dropna()
    if not sub.empty:
        fig.add_trace(go.Box(y=sub.values, name=trim,
            marker_color="#7eb8f7", line=dict(color="#7eb8f7", width=1.2),
            fillcolor="rgba(126,184,247,0.12)", boxpoints=False), row=1, col=1)

# Panel derecho: RETORNOS
for trim in trims:
    sub = df2[df2["trim"]==trim]["retorno"].dropna()
    if not sub.empty:
        fig.add_trace(go.Box(y=sub.values, name=trim,
            marker_color="#88e0b4", line=dict(color="#88e0b4", width=1.2),
            fillcolor="rgba(136,224,180,0.12)", boxpoints=False), row=1, col=2)

fig.add_hline(y=0, line_dash="dot", line_color="#484f58",
              line_width=1, row=1, col=2)

fig.update_layout(height=420, paper_bgcolor="#161b22", plot_bgcolor="#161b22",
    font_color="#8b949e", title_text="Estacionariedad: Boxplot por trimestre — BTC",
    title_font_color="#e6edf3", showlegend=False,
    margin=dict(t=60, b=30, l=60, r=20))
for r,c in [(1,1),(1,2)]:
    fig.update_xaxes(row=r,col=c,gridcolor="#1c2230",tickfont=dict(color="#8b949e",size=8),tickangle=-30)
    fig.update_yaxes(row=r,col=c,gridcolor="#1c2230",tickfont=dict(color="#8b949e"))
fig.update_yaxes(row=1,col=1,tickprefix="$",tickformat=",")
fig.update_yaxes(row=1,col=2,ticksuffix="%")
for ann in fig.layout.annotations:
    ann.font.color = "#e6edf3"
    ann.font.size  = 12
fig.show()

print("\n INTERPRETACIÓN:")
print("   Precios: cada caja está a un nivel diferente → media cambia → NO estacionaria")
print("   Retornos: cajas al mismo nivel (≈0%) → media constante → SÍ estacionaria")



 INTERPRETACIÓN:
   Precios: cada caja está a un nivel diferente → media cambia → NO estacionaria
   Retornos: cajas al mismo nivel (≈0%) → media constante → SÍ estacionaria



<a id="decomp"></a>
## 7. Descomposición de la Serie Temporal

### 7.1 Comparación: Aditiva vs Multiplicativa


In [11]:

from statsmodels.tsa.seasonal import seasonal_decompose

serie  = df_btc.set_index("date")["close"].dropna()
period = 30   # ciclo mensual aproximado

print("Descomponiendo la serie de precios BTC...")
print(f"   Observaciones: {len(serie)}")
print(f"   Período ciclo: {period} días\n")

for model_type, color_t, color_s, color_r, label in [
    ("additive",       "#7eb8f7","#f7c87e","#f7958a","Aditivo  (Y = T + S + R)"),
    ("multiplicative", "#88e0b4","#c5b8f0","#f7a8c4","Multiplicativo  (Y = T × S × R)"),
]:
    result = seasonal_decompose(serie, model=model_type, period=period, extrapolate_trend="freq")

    fig = make_subplots(rows=4,cols=1,shared_xaxes=True,
        subplot_titles=["Serie original","Tendencia (T)","Estacionalidad (S)","Residuo (R)"],
        vertical_spacing=0.06)

    for i,(data,color,fill) in enumerate([
        (serie,              "#e6edf3", None),
        (result.trend,       color_t,   None),
        (result.seasonal,    color_s,   None),
        (result.resid,       color_r,   "tozeroy"),
    ], 1):
        kw = dict(
            fill=fill,
            fillcolor=f"rgba({int(color[1:3],16)},{int(color[3:5],16)},{int(color[5:7],16)},0.12)"
        ) if fill else {}
        fig.add_trace(go.Scatter(x=data.index,y=data.values,mode="lines",
            line=dict(color=color,width=1.4),showlegend=False,**kw), row=i,col=1)

    fig.update_layout(height=550,paper_bgcolor="#161b22",plot_bgcolor="#161b22",
        font_color="#8b949e",
        title_text=f"Descomposición {label}  —  BTC/USD",
        title_font_color="#e6edf3",
        margin=dict(t=60,b=24,l=52,r=20))
    for r in range(1,5):
        fig.update_xaxes(row=r,gridcolor="#1c2230",zeroline=False,tickfont=dict(color="#8b949e",size=8))
        fig.update_yaxes(row=r,gridcolor="#1c2230",zeroline=False,tickfont=dict(color="#8b949e",size=8))
    for ann in fig.layout.annotations:
        ann.font.color="#e6edf3"; ann.font.size=10
    fig.show()

    # Estadísticas de componentes
    var_t = float(result.trend.dropna().var())
    var_s = float(result.seasonal.dropna().var())
    var_r = float(result.resid.dropna().var())
    var_total = var_t + var_s + var_r
    if var_total == 0: var_total = 1
    print(f" Modelo {model_type.upper()} — Varianza explicada por componente:")
    print(f"   Tendencia:      {var_t/var_total*100:.1f}%")
    print(f"   Estacionalidad: {var_s/var_total*100:.1f}%")
    print(f"   Residuo:        {var_r/var_total*100:.1f}%\n")


Descomponiendo la serie de precios BTC...
   Observaciones: 366
   Período ciclo: 30 días



 Modelo ADDITIVE — Varianza explicada por componente:
   Tendencia:      97.0%
   Estacionalidad: 0.1%
   Residuo:        2.9%



 Modelo MULTIPLICATIVE — Varianza explicada por componente:
   Tendencia:      100.0%
   Estacionalidad: 0.0%
   Residuo:        0.0%




### 7.2 ¿Cuál modelo es más apropiado para criptomonedas?

| Criterio | Aditivo | Multiplicativo |
|----------|---------|----------------|
| Amplitud de ciclos | Constante | Crece con el precio |
| Unidad de componentes | USD | Factor (ratio) |
| Aplicación | Retornos logarítmicos | Precios originales |
| En cripto |  Para retornos |  Para precios |

**Conclusión:** Los precios de Bitcoin siguen un modelo **multiplicativo** porque 
un movimiento del 5% representa un monto en USD que crece proporcionalmente al nivel 
del precio. Si se trabaja con retornos logarítmicos, la serie se vuelve aditiva.



<a id="corr"></a>
## 8. Correlaciones entre Criptomonedas

### 8.1 Construcción de la matriz de correlaciones


In [12]:

# ── Obtener histórico de múltiples monedas ───────────────────────────────────

COINS = ["BTC","ETH","BNB","SOL","XRP","ADA","AVAX","DOGE","LTC","LINK"]

print("Obteniendo histórico de las 10 monedas (puede tomar unos segundos)...")
frames = {}
for sym in COINS:
    df_sym = get_histoday(sym, days=365)
    if not df_sym.empty:
        s = df_sym.set_index("date")["close"].replace(0,np.nan)
        frames[sym] = s

# Unir en DataFrame wide e interpolar NaN
df_wide = pd.DataFrame(frames).interpolate(method="linear").dropna()
print(f"\n DataFrame wide: {df_wide.shape[0]} días × {df_wide.shape[1]} monedas")
print(f"   Rango temporal: {df_wide.index[0].date()} → {df_wide.index[-1].date()}")

# Correlación de retornos
rets_wide = df_wide.pct_change().dropna()
corr      = rets_wide.corr().round(3)

print(f"\n Correlación media entre monedas: {corr.values[np.triu_indices_from(corr.values,k=1)].mean():.3f}")
print(f"   Par más correlacionado: ", end="")
vals = corr.copy()
np.fill_diagonal(vals.values, 0)
idx  = np.unravel_index(vals.values.argmax(), vals.shape)
print(f"{vals.index[idx[0]]} ↔ {vals.columns[idx[1]]}: r={vals.values[idx]:.3f}")


Obteniendo histórico de las 10 monedas (puede tomar unos segundos)...
 Histórico BTC desde API: 366 días
 Histórico ETH desde API: 366 días
 Histórico BNB desde API: 366 días
 Histórico SOL desde API: 366 días
 Histórico XRP desde API: 366 días
 Histórico ADA desde API: 366 días
 Histórico AVAX desde API: 366 días
 Histórico DOGE desde API: 366 días
 Histórico LTC desde API: 366 días
 Histórico LINK sintético generado: 366 días

 DataFrame wide: 731 días × 10 monedas
   Rango temporal: 2025-05-12 → 2026-05-12

 Correlación media entre monedas: 0.625
   Par más correlacionado: ADA ↔ DOGE: r=0.893


In [13]:

# ── Heatmap de correlaciones ─────────────────────────────────────────────────

fig = px.imshow(corr, text_auto=True,
    color_continuous_scale=[[0,"#c0392b"],[0.5,"#161b22"],[1,"#27ae60"]],
    zmin=-1, zmax=1,
    title="Correlación de Pearson (r) — Retornos diarios, 1 año")
fig.update_traces(textfont=dict(size=10))
fig.update_layout(height=500,paper_bgcolor="#161b22",
    font_color="#8b949e",title_font_color="#e6edf3",
    margin=dict(t=60,b=20,l=20,r=20),
    coloraxis_colorbar=dict(tickfont=dict(color="#8b949e"),
        title=dict(text="r",font=dict(color="#8b949e")),thickness=12))
fig.update_xaxes(tickfont=dict(color="#e6edf3",size=10))
fig.update_yaxes(tickfont=dict(color="#e6edf3",size=10))
fig.show()

print("\n INTERPRETACIÓN:")
print("   Verde intenso (r≈1.0): monedas que se mueven juntas")
print("   Rojo intenso (r≈-1.0): monedas que se mueven al revés")
print("   La diagonal siempre es 1.0 (cada moneda correlaciona consigo misma)")
print("   Alta correlación general = mercado dominado por el ciclo de BTC")



 INTERPRETACIÓN:
   Verde intenso (r≈1.0): monedas que se mueven juntas
   Rojo intenso (r≈-1.0): monedas que se mueven al revés
   La diagonal siempre es 1.0 (cada moneda correlaciona consigo misma)
   Alta correlación general = mercado dominado por el ciclo de BTC



<a id="arima"></a>
## 9. Predicción con Modelo ARIMA

### 9.1 Metodología ARIMA


In [14]:

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import pacf as _pacf

warnings.filterwarnings("ignore")

# ── Preparar serie ────────────────────────────────────────────────────────────
serie_btc = df_btc.set_index("date")["close"].replace(0,np.nan).dropna()

# Separar validación (últimos 30 días)
val_size  = 30
train_end = len(serie_btc) - val_size
train_s   = serie_btc.iloc[:train_end]
val_s     = serie_btc.iloc[train_end:]

print(f" Serie BTC:")
print(f"   Total observaciones:  {len(serie_btc)}")
print(f"   Entrenamiento:        {len(train_s)} días")
print(f"   Validación:           {len(val_s)} días")

# ── Ajustar ARIMA(2,1,2) ─────────────────────────────────────────────────────
order  = (2, 1, 2)
model  = ARIMA(train_s, order=order)
fitted = model.fit()

print(f"\n Modelo ARIMA{order} ajustado:")
print(f"   AIC: {fitted.aic:.2f}")
print(f"   BIC: {fitted.bic:.2f}")
print(f"   Log-verosimilitud: {fitted.llf:.2f}")


 Serie BTC:
   Total observaciones:  366
   Entrenamiento:        336 días
   Validación:           30 días

 Modelo ARIMA(2, 1, 2) ajustado:
   AIC: 6057.77
   BIC: 6076.84
   Log-verosimilitud: -3023.89


In [15]:

# ── Validación y pronóstico ───────────────────────────────────────────────────

# Validación
val_fc   = fitted.forecast(steps=val_size)
mae      = float(np.mean(np.abs(val_fc - val_s.values)))
rmse     = float(np.sqrt(np.mean((val_fc - val_s.values)**2)))
mape     = float(np.mean(np.abs((val_fc - val_s.values) / np.clip(np.abs(val_s.values),1e-9,None))) * 100)

print(" Métricas de validación (últimos 30 días):")
print(f"   MAE:   ${mae:,.2f}  — Error absoluto medio en USD")
print(f"   RMSE:  ${rmse:,.2f}  — Raíz del error cuadrático medio")
print(f"   MAPE:  {mape:.2f} %  — Error porcentual absoluto medio")

# Pronóstico futuro 14 días
horizon  = 14
fc       = fitted.get_forecast(steps=horizon)
fc_mean  = fc.predicted_mean
ci95     = fc.conf_int(alpha=0.05)
ci80     = fc.conf_int(alpha=0.20)
fc_dates = pd.date_range(serie_btc.index[-1] + pd.Timedelta(days=1),
                          periods=horizon, freq="D")

# Prueba Ljung-Box
resid    = fitted.resid.dropna()
lb       = acorr_ljungbox(resid, lags=[10], return_df=True)
lb_p     = float(lb["lb_pvalue"].iloc[0])

print(f"\n Pronóstico ARIMA{order} — próximos {horizon} días:")
print(f"   Precio actual:        ${serie_btc.iloc[-1]:,.2f}")
print(f"   Precio pronosticado:  ${float(fc_mean.iloc[-1]):,.2f}")
cambio = (float(fc_mean.iloc[-1]) / serie_btc.iloc[-1] - 1) * 100
print(f"   Cambio esperado:      {'+' if cambio>=0 else ''}{cambio:.1f}%")
print(f"\n Ljung-Box (independencia residuos):")
print(f"   p-valor: {lb_p:.4f} → {' Residuos independientes (modelo adecuado)' if lb_p>0.05 else ' Hay estructura residual'}")


 Métricas de validación (últimos 30 días):
   MAE:   $7,167.08  — Error absoluto medio en USD
   RMSE:  $7,545.40  — Raíz del error cuadrático medio
   MAPE:  9.12 %  — Error porcentual absoluto medio

 Pronóstico ARIMA(2, 1, 2) — próximos 14 días:
   Precio actual:        $80,742.40
   Precio pronosticado:  $70,686.78
   Cambio esperado:      -12.5%

 Ljung-Box (independencia residuos):
   p-valor: 1.0000 →  Residuos independientes (modelo adecuado)


In [16]:

# ── Gráfico de pronóstico con intervalos de confianza ────────────────────────

n_show = min(120, len(train_s))
color_fc = "#88e0b4" if cambio >= 0 else "#f7958a"

fig = go.Figure()

# Histórico
fig.add_trace(go.Scatter(
    x=train_s.index[-n_show:], y=train_s.values[-n_show:],
    mode="lines", name="Histórico",
    line=dict(color="#484f58", width=1.2)))

# Validación real vs ARIMA
fig.add_trace(go.Scatter(
    x=val_s.index, y=val_s.values,
    mode="lines", name="Real (validación)",
    line=dict(color="#e6edf3", width=1.5)))

fig.add_trace(go.Scatter(
    x=val_s.index, y=val_fc,
    mode="lines", name="ARIMA (validación)",
    line=dict(color="#7eb8f7", width=1.5, dash="dot")))

# IC 95%
fig.add_trace(go.Scatter(
    x=list(fc_dates)+list(reversed(fc_dates)),
    y=list(ci95.iloc[:,1])+list(reversed(ci95.iloc[:,0])),
    fill="toself", fillcolor="rgba(126,184,247,0.12)",
    line=dict(color="rgba(0,0,0,0)"), name="IC 95%", hoverinfo="skip"))

# IC 80%
fig.add_trace(go.Scatter(
    x=list(fc_dates)+list(reversed(fc_dates)),
    y=list(ci80.iloc[:,1])+list(reversed(ci80.iloc[:,0])),
    fill="toself", fillcolor="rgba(126,184,247,0.22)",
    line=dict(color="rgba(0,0,0,0)"), name="IC 80%", hoverinfo="skip"))

# Pronóstico central
fig.add_trace(go.Scatter(
    x=fc_dates, y=fc_mean.values,
    mode="lines+markers", name=f"Pronóstico {horizon}d",
    line=dict(color=color_fc, width=2),
    marker=dict(size=4, color=color_fc)))

fig.add_vline(x=str(fc_dates[0]), line_dash="dot",
              line_color="#484f58", line_width=1)

fig.update_layout(height=450, paper_bgcolor="#161b22", plot_bgcolor="#161b22",
    font_color="#8b949e",
    title_text=f"BTC/USD — ARIMA{order} · Pronóstico {horizon} días",
    title_font_color="#e6edf3",
    legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(color="#8b949e")),
    hovermode="x unified", margin=dict(t=55,b=30,l=60,r=20))
fig.update_xaxes(gridcolor="#1c2230", tickfont=dict(color="#8b949e"))
fig.update_yaxes(gridcolor="#1c2230", tickprefix="$", tickformat=",",
                 tickfont=dict(color="#8b949e"))
fig.show()

print("\n Zona azul clara = IC 95%: hay 95% de probabilidad de que el precio real")
print("   esté dentro de ese rango en el horizonte de pronóstico.")
print(" Zona azul media = IC 80%: rango más estrecho con 80% de confianza.")



 Zona azul clara = IC 95%: hay 95% de probabilidad de que el precio real
   esté dentro de ese rango en el horizonte de pronóstico.
 Zona azul media = IC 80%: rango más estrecho con 80% de confianza.


In [17]:

# ── Diagnósticos del modelo ARIMA ────────────────────────────────────────────

fig = make_subplots(rows=2,cols=2,
    subplot_titles=["Residuos en el tiempo","ACF de residuos",
                    "PACF de residuos","QQ-Plot"])

# Residuos
fig.add_trace(go.Scatter(x=list(range(len(resid))),y=resid.values,
    mode="lines",line=dict(color="#7eb8f7",width=0.8),name="Residuos"),row=1,col=1)
fig.add_hline(y=0,line_dash="dot",line_color="#484f58",line_width=1,row=1,col=1)

# ACF
nlags = min(40, len(resid)//3)
cb    = 1.96/np.sqrt(len(resid))
acf_v = [float(resid.autocorr(lag=i)) for i in range(1,nlags+1)]
fig.add_trace(go.Bar(x=list(range(1,nlags+1)),y=acf_v,
    marker_color="#7eb8f7",opacity=0.75,name="ACF"),row=1,col=2)
fig.add_hline(y= cb,line_dash="dot",line_color="#484f58",line_width=1,row=1,col=2)
fig.add_hline(y=-cb,line_dash="dot",line_color="#484f58",line_width=1,row=1,col=2)

# PACF
try:
    pv = _pacf(resid, nlags=nlags, method="ywm")
    pacf_v = list(pv[1:])
except Exception:
    pacf_v = [0.0]*nlags
fig.add_trace(go.Bar(x=list(range(1,nlags+1)),y=pacf_v,
    marker_color="#88e0b4",opacity=0.75,name="PACF"),row=2,col=1)
fig.add_hline(y= cb,line_dash="dot",line_color="#484f58",line_width=1,row=2,col=1)
fig.add_hline(y=-cb,line_dash="dot",line_color="#484f58",line_width=1,row=2,col=1)

# QQ-Plot manual
n_r  = len(resid)
emp  = np.sort(resid.values)
rng2 = np.random.default_rng(42)
theo = np.array([float(np.percentile(rng2.standard_normal(10000),(i/(n_r+1))*100))
                 for i in range(1,n_r+1)])
q25e,q75e = np.percentile(emp,25),np.percentile(emp,75)
q25t,q75t = np.percentile(theo,25),np.percentile(theo,75)
slope = (q75e-q25e)/max(q75t-q25t,1e-9)
inter = q25e - slope*q25t
xl    = np.array([theo.min(),theo.max()])
fig.add_trace(go.Scatter(x=theo,y=emp,mode="markers",
    marker=dict(color="#7eb8f7",size=3,opacity=0.55),name="Quantiles"),row=2,col=2)
fig.add_trace(go.Scatter(x=xl,y=slope*xl+inter,mode="lines",
    line=dict(color="#f7958a",width=1.5,dash="dot"),name="Referencia"),row=2,col=2)

fig.update_layout(height=480,paper_bgcolor="#161b22",plot_bgcolor="#161b22",
    font_color="#8b949e",title_text=f"Diagnósticos ARIMA{order}",
    title_font_color="#e6edf3",showlegend=False,margin=dict(t=60,b=20,l=40,r=20))
for r in [1,2]:
    for c in [1,2]:
        fig.update_xaxes(row=r,col=c,gridcolor="#1c2230",zeroline=False,tickfont=dict(color="#8b949e",size=7))
        fig.update_yaxes(row=r,col=c,gridcolor="#1c2230",zeroline=False,tickfont=dict(color="#8b949e",size=7))
for ann in fig.layout.annotations:
    ann.font.color="#e6edf3"; ann.font.size=10
fig.show()

print("\n Cómo leer los diagnósticos:")
print("   Residuos: deben parecer ruido aleatorio sin patrones → modelo capturó la estructura")
print("   ACF/PACF: barras dentro de las líneas punteadas → no hay autocorrelación residual")
print("   QQ-Plot:  puntos cerca de la línea roja → residuos aproximadamente normales")



 Cómo leer los diagnósticos:
   Residuos: deben parecer ruido aleatorio sin patrones → modelo capturó la estructura
   ACF/PACF: barras dentro de las líneas punteadas → no hay autocorrelación residual
   QQ-Plot:  puntos cerca de la línea roja → residuos aproximadamente normales



<a id="conclusiones"></a>
## 10. Conclusiones

### 10.1 Hallazgos Principales

| # | Hallazgo | Evidencia |
|---|----------|-----------|
| 1 | **Bitcoin domina el mercado** | Dominancia consistente >40%. Sus movimientos arrastran al resto. |
| 2 | **Precios NO estacionarios** | ADF p>0.05 en todos los pares. Se requiere d=1 para ARIMA. |
| 3 | **Retornos SÍ estacionarios** | ADF p<0.05 en retornos. Media ≈ 0% estable por trimestres. |
| 4 | **Leptocurtosis en retornos** | Curtosis >3 en todas las monedas. Más eventos extremos que la normal. |
| 5 | **Alta correlación entre monedas** | r promedio 0.6–0.9. Siguen el ciclo de BTC (efecto contagio). |
| 6 | **Modelo multiplicativo más apropiado** | Las oscilaciones son proporcionales al precio, no absolutas. |
| 7 | **ARIMA viable como baseline** | MAPE razonable en validación, superando el modelo naive. |

### 10.2 Limitaciones

- **API gratuita:** límite de ~50 req/hora; el fallback sintético mantiene funcionalidad.
- **ARIMA no capta volatilidad variable:** modelos GARCH serían más apropiados para la varianza.
- **No incorpora factores externos:** noticias, regulaciones y sentimiento de mercado.
- **Estacionalidad débil:** los mercados cripto operan 24/7 sin feriados ni ciclos estrictos.

---

> ⚠️ **Aviso:** Este análisis es de naturaleza académica. La información presentada 
> no constituye asesoramiento financiero. Invertir en criptomonedas implica riesgo 
> de pérdida total del capital.

---

*Desarrollado con Python 3.11 · Dash · Plotly · Statsmodels · SciPy · CryptoCompare API*


In [18]:

# ── Resumen ejecutivo automático ─────────────────────────────────────────────

print("=" * 65)
print("  RESUMEN EJECUTIVO — Crypto Market Dashboard")
print("=" * 65)

# Snapshot del mercado
total_mcap = df_snap["market_cap"].sum()
btc_dom    = df_snap.loc[df_snap["symbol"]=="BTC","market_cap"].sum() / total_mcap * 100
avg_chg    = df_snap["change_pct_24h"].mean()

print(f"\n MERCADO (snapshot actual):")
print(f"   Cap. total:      {fmt_usd(total_mcap)}")
print(f"   Dominancia BTC:  {btc_dom:.1f}%")
print(f"   Cambio prom 24h: {avg_chg:+.2f}%")

# Estadísticas BTC
ret_anual  = float(df_btc["close"].pct_change().std() * np.sqrt(365) * 100)
rend_1a    = (df_btc["close"].iloc[-1] / df_btc["close"].iloc[0] - 1) * 100
print(f"\n₿ BITCOIN (1 año):")
print(f"   Precio actual:     ${df_btc['close'].iloc[-1]:,.2f}")
print(f"   Rendimiento 1 año: {rend_1a:+.1f}%")
print(f"   Volatilidad anual: {ret_anual:.1f}%")
print(f"   ADF precios:       p={adf_p[1]:.4f} → NO estacionaria")
print(f"   ADF retornos:      p={adf_r[1]:.4f} → SÍ estacionaria")

# ARIMA
print(f"\n ARIMA{order}:")
print(f"   AIC:    {fitted.aic:.2f}")
print(f"   MAE:    ${mae:,.2f}")
print(f"   MAPE:   {mape:.2f}%")
print(f"   Pronóstico +{horizon}d: ${float(fc_mean.iloc[-1]):,.2f} ({'+' if cambio>=0 else ''}{cambio:.1f}%)")
print(f"   Ljung-Box p={lb_p:.4f}: {' OK' if lb_p>0.05 else ' Revisar'}")

print("\n" + "=" * 65)


  RESUMEN EJECUTIVO — Crypto Market Dashboard

 MERCADO (snapshot actual):
   Cap. total:      $2.80T
   Dominancia BTC:  69.6%
   Cambio prom 24h: +3.31%

₿ BITCOIN (1 año):
   Precio actual:     $80,742.40
   Rendimiento 1 año: -21.5%
   Volatilidad anual: 42.2%
   ADF precios:       p=0.7794 → NO estacionaria
   ADF retornos:      p=0.0000 → SÍ estacionaria

 ARIMA(2, 1, 2):
   AIC:    6057.77
   MAE:    $7,167.08
   MAPE:   9.12%
   Pronóstico +14d: $70,686.78 (-12.5%)
   Ljung-Box p=1.0000:  OK

